# 1.Librerías

In [1]:
import warnings
warnings.filterwarnings('ignore')
import openpyxl
import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None
import os
import sys
import time
import pyodbc

In [2]:
!pip show lightgbm

Name: lightgbm
Version: 4.5.0
Summary: LightGBM Python Package
Home-page: https://github.com/microsoft/LightGBM
Author: 
Author-email: 
License: The MIT License (MIT)

Copyright (c) Microsoft Corporation

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS O

In [3]:
# export http_proxy=http://proxyprd.scotia-capital.com:8080
# export https_proxy=http://proxyprd.scotia-capital.com:8080
# pip install lightgbm==4.5.0 --trusted-host pypi.org --trusted-host pypi.python.org --trusted-host files.pythonhosted.org
!pip install lightgbm==4.5.0 --proxy http://latam-proxy.glb.lnm.bns:8080

  Using cached lightgbm-4.5.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.5.0-py3-none-win_amd64.whl (1.4 MB)
  Attempting uninstall: lightgbm
    Found existing installation: lightgbm 3.3.5
    Uninstalling lightgbm-3.3.5:
      Successfully uninstalled lightgbm-3.3.5



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Directorio principal
ruta_base = "C:/Users/s5226347/OneDrive - The Bank of Nova Scotia/Pedidos/023. Estimador_Ingresos_Cliente"
#ruta_base = "C:/Users/s2660778/The Bank of Nova Scotia/Gonzales Cornejo, Kevin Fausto - 023. Estimador_Ingresos_Cliente"
os.chdir(ruta_base+"/Sprint 4/Single_Model_VF")
sys.path.append(ruta_base+'/Librerias')
import sbp_modelos as sbp

# 2.Importar base

Desde csv o parquet:

In [4]:
#df_impl = pd.read_parquet(ruta_base+"/Entregables_Validacion/Entregables_Iniciales/Bases/MALBIE2_RCC_202507.parquet",engine="pyarrow")
df_impl = pd.read_csv(ruta_base+'/Bases/MALBIE2_RCC_202601.txt')

In [ ]:
#df_impl = df_train.copy()

Desde SQL:

In [4]:
#CONEXION AL SERVIDOR 13
server = 'CGSSCSF13\\SQL3,277'
bd = 'MODELOSH'
conexion = pyodbc.connect(driver='SQL server', host = server, database = bd)
print('Conexión exitosa')
query = f'''(
    SELECT --SELECT TOP 100
        -- VARIABLES IDENTIFICADORAS:
        ID_REGISTRO,
        -- VARIABLES INPUT:
        GHIGHESTBAL_NORM48,MED_PAY_24M,UBIGEO_INMUEBLE,GAVGCCLIMIT1TO48,CARRERA,MAX_CONSUMO_TOP4_60M,MAX_HIPOTECARIO_2010,MAX_MAX_LINCRE_TOP4_12M,
        MED_PASIVOS_IC_MIN6_12M,MONTO_TRANSACCIONES_12M,MED_RED_CONSUMO_NREV_24M,AVG_RED_CONSUMO_TOTAL_24M,GBAL1PCTAVGBAL1TO18,MAX_MNTO_ENV_PLIN_12M,
        MAX_COMPRAS_TC_24M,MED_RCC_S_TOT_U12M_6CONS,ANTIGUEDAD_TCPP_144M,MAX_GRUPO_VEHICULO,NUM_PASIVOS_IC_12M,MAX_GARHIP_120M
    FROM MODELOSH.DBO.MALBIE2_RCC_202502
) '''
df_impl = pd.read_sql_query(query, conexion)
conexion.close()

Conexión exitosa


# 3.Tratamiento de variables

In [5]:
# Renombra col MAX_MAX_LINCRE_TOP4_24M solo para la prediccion
if 'MAX_MAX_LINCRE_TOP4_12M' in df_impl.columns:
    df_impl = df_impl.rename(columns={'MAX_MAX_LINCRE_TOP4_12M':'MAX_MAX_LINCRE_TOP4_24M'})

# Valores especiales a missing
df_impl = sbp.convertir_valores_especiales_a_missing(df_impl)

# Tratamiento de MAX_GRUPO_VEHICULO
mapping = {99: 24}
if (df_impl['MAX_GRUPO_VEHICULO'] == 99).any():
    df_impl['MAX_GRUPO_VEHICULO'] = df_impl['MAX_GRUPO_VEHICULO'].replace(mapping)

# Imputamos NaN por la categoría 'Missing' y convertimos los tipos de variables nominales
lista_var_nominales = ['UBIGEO_INMUEBLE','CARRERA']
for i in lista_var_nominales:
  df_impl[i] = df_impl[i].fillna('Missing')
  df_impl[i] = df_impl[i].astype('category')

# Winsorización
features_malbie2={
 'GHIGHESTBAL_NORM48':[None, 4100083]
,'MED_PAY_24M':[0.03, 352900.7]
,'MAX_CONSUMO_TOP4_60M':[None, 2199520]
,'MED_PASIVOS_IC_MIN6_12M':[50, 6744760]
,'MAX_MAX_LINCRE_TOP4_24M':[108.96, 383100]
,'GAVGCCLIMIT1TO48':[112.58, 2970559]
,'MONTO_TRANSACCIONES_12M':[-151653.61, 180162.2]
,'AVG_RED_CONSUMO_TOTAL_24M':[50, 286641.4]
,'MAX_HIPOTECARIO_2010':[None, 8540948]
,'MED_RED_CONSUMO_NREV_24M':[50, 205768.4]
,'MAX_MNTO_ENV_PLIN_12M':[1, 27826.8]
,'MAX_COMPRAS_TC_24M':[None, 284622.6]
,'MAX_GARHIP_120M':[None, 28416330]
,'GBAL1PCTAVGBAL1TO18':[0.16, 1800]
,'MED_RCC_S_TOT_U12M_6CONS':[54.34, 56131.9]
}
for col, percentil in features_malbie2.items():
    df_impl.loc[df_impl[col] < percentil[0], col] = percentil[0]
    df_impl.loc[df_impl[col] > percentil[1], col] = percentil[1]


# 4.Modelamiento

In [6]:
features_malbie2=['UBIGEO_INMUEBLE','CARRERA','GHIGHESTBAL_NORM48','MED_PAY_24M','MAX_CONSUMO_TOP4_60M','MED_PASIVOS_IC_MIN6_12M',
        'MONTO_TRANSACCIONES_12M','MED_RED_CONSUMO_NREV_24M','MAX_MAX_LINCRE_TOP4_24M','MAX_HIPOTECARIO_2010',
        'AVG_RED_CONSUMO_TOTAL_24M','GAVGCCLIMIT1TO48','ANTIGUEDAD_TCPP_144M','GBAL1PCTAVGBAL1TO18','MED_RCC_S_TOT_U12M_6CONS',
        'MAX_MNTO_ENV_PLIN_12M','MAX_GRUPO_VEHICULO','MAX_GARHIP_120M','MAX_COMPRAS_TC_24M']

features_malbie2_no_cat=['GHIGHESTBAL_NORM48','MED_PAY_24M','MAX_CONSUMO_TOP4_60M','MED_PASIVOS_IC_MIN6_12M',
        'MONTO_TRANSACCIONES_12M','MED_RED_CONSUMO_NREV_24M','MAX_MAX_LINCRE_TOP4_24M','MAX_HIPOTECARIO_2010',
        'AVG_RED_CONSUMO_TOTAL_24M','GAVGCCLIMIT1TO48','ANTIGUEDAD_TCPP_144M','GBAL1PCTAVGBAL1TO18',
        'MED_RCC_S_TOT_U12M_6CONS','MAX_MNTO_ENV_PLIN_12M','MAX_GRUPO_VEHICULO','MAX_GARHIP_120M','MAX_COMPRAS_TC_24M']

features_malbie2_cat = ['UBIGEO_INMUEBLE','CARRERA']

X = df_impl[features_malbie2]

In [7]:
# Cargar el modelo entrenado
import joblib
lgbm_model = joblib.load(ruta_base+'/Entregables_Validacion/Entregables_Iniciales/Artefactos/MODEL_lgbm_Con_Pesos_Final_vf3.pkl')
sbp.importancias_lgbm_v2(lgbm_model,X)

,Var,Split_Importance,Gain_Importance,Split_Weight,Gain_Weight,Avg_Weight,H_Avg_Weight
0,GHIGHESTBAL_NORM48,4604,15201.263187,0.093159,0.251503,0.172331,0.172423
1,MED_PAY_24M,3021,8048.648344,0.061128,0.133164,0.097146,0.106266
2,CARRERA,3549,4599.521312,0.071812,0.076099,0.073955,0.093712
3,MAX_CONSUMO_TOP4_60M,2927,4258.549230,0.059226,0.070457,0.064842,0.081616
4,MED_PASIVOS_IC_MIN6_12M,2372,3644.881956,0.047996,0.060304,0.054150,0.067787
5,MAX_MAX_LINCRE_TOP4_24M,2329,3566.517483,0.047126,0.059008,0.053067,0.066456
6,AVG_RED_CONSUMO_TOTAL_24M,2844,2781.882597,0.057546,0.046026,0.051786,0.064863
7,GAVGCCLIMIT1TO48,2692,2702.749663,0.054471,0.044717,0.049594,0.062287
8,MONTO_TRANSACCIONES_12M,2705,2358.874546,0.054734,0.039027,0.046881,0.057786
9,MAX_HIPOTECARIO_2010,1045,7547.780137,0.021145,0.124877,0.073011,0.045866


In [ ]:
# Las importancias tienen que cuadrar con el siguiente cuadro:

,Var,Split_Importance,Gain_Importance,Split_Weight,Gain_Weight,Avg_Weight,H_Avg_Weight
0,GHIGHESTBAL_NORM48,4604,15201.263187,0.093159,0.251503,0.172331,0.172423
1,MED_PAY_24M,3021,8048.648344,0.061128,0.133164,0.097146,0.106266
2,CARRERA,3549,4599.521312,0.071812,0.076099,0.073955,0.093712
3,MAX_CONSUMO_TOP4_60M,2927,4258.549230,0.059226,0.070457,0.064842,0.081616
4,MED_PASIVOS_IC_MIN6_12M,2372,3644.881956,0.047996,0.060304,0.054150,0.067787
5,MAX_MAX_LINCRE_TOP4_24M,2329,3566.517483,0.047126,0.059008,0.053067,0.066456
6,AVG_RED_CONSUMO_TOTAL_24M,2844,2781.882597,0.057546,0.046026,0.051786,0.064863
7,GAVGCCLIMIT1TO48,2692,2702.749663,0.054471,0.044717,0.049594,0.062287
8,MONTO_TRANSACCIONES_12M,2705,2358.874546,0.054734,0.039027,0.046881,0.057786
9,MAX_HIPOTECARIO_2010,1045,7547.780137,0.021145,0.124877,0.073011,0.045866


#### Prediccion

In [8]:
df_impl["INGRESO_MALBIE_V2"] = np.exp(lgbm_model.predict(X))

In [9]:
# Valida si hay missings en la prediccion
df_impl[df_impl.isna()].count()

ID_REGISTRO                  0
GHIGHESTBAL_NORM48           0
MED_PAY_24M                  0
UBIGEO_INMUEBLE              0
GAVGCCLIMIT1TO48             0
CARRERA                      0
MAX_CONSUMO_TOP4_60M         0
MAX_HIPOTECARIO_2010         0
MAX_MAX_LINCRE_TOP4_24M      0
MED_PASIVOS_IC_MIN6_12M      0
MONTO_TRANSACCIONES_12M      0
MED_RED_CONSUMO_NREV_24M     0
AVG_RED_CONSUMO_TOTAL_24M    0
GBAL1PCTAVGBAL1TO18          0
MAX_MNTO_ENV_PLIN_12M        0
MAX_COMPRAS_TC_24M           0
MED_RCC_S_TOT_U12M_6CONS     0
ANTIGUEDAD_TCPP_144M         0
MAX_GRUPO_VEHICULO           0
NUM_PASIVOS_IC_12M           0
MAX_GARHIP_120M              0
INGRESO_MALBIE_V2            0
dtype: int64

#### Export

In [10]:
# Export
features_exp = ["ID_REGISTRO","INGRESO_MALBIE_V2"]
df_impl["INGRESO_MALBIE_V2"] = np.round(df_impl["INGRESO_MALBIE_V2"],2)
df_impl[features_exp].to_csv(ruta_base+"/output_RCC_202502.csv", index=False)
print(ruta_base)

C:/Users/s5226347/OneDrive - The Bank of Nova Scotia/Pedidos/023. Estimador_Ingresos_Cliente


#### Metricas

In [9]:
bins = [999,1500,2000,3500,5000,7000,10000,12500,15000,float("inf")]
sbp.calcular_precision_x_intervalos_v3(df_impl["INGRESO_MALBIE_V2"],df_impl["INGRESO_CAP_V2"],0.30,bins)

,Casos,Casos (%),Prom. real,Prom. pred,MAE,MAPE,RMSPE,Subestimacion,Coincidencia,Sobreestimacion
Intervalos,,,,,,,,,,
"(999.0, 1500.0]",115036,0.223984,1252.434248,1540.508490,317.185497,0.264855,0.483425,0.000000,0.736561,0.263439
"(1500.0, 2000.0]",97665,0.190161,1733.512042,1739.131861,388.516519,0.221816,0.362392,0.031905,0.843414,0.124681
"(2000.0, 3500.0]",154038,0.299923,2662.076177,2428.753122,773.545547,0.289841,0.386179,0.328847,0.555499,0.115653
"(3500.0, 5000.0]",57509,0.111974,4166.864644,3597.333313,1413.813411,0.339227,0.430070,0.382392,0.498948,0.118660
"(5000.0, 7000.0]",37677,0.073360,5880.914725,5118.507553,1914.197782,0.325943,0.416129,0.348515,0.535658,0.115827
"(7000.0, 10000.0]",25254,0.049171,8308.271892,6941.771972,2726.753720,0.327394,0.409025,0.374436,0.526768,0.098796
"(10000.0, 12500.0]",9664,0.018817,11161.192310,8853.708644,3798.050362,0.339954,0.414846,0.424048,0.495344,0.080608
"(12500.0, 15000.0]",5704,0.011106,13686.566617,10512.281662,4786.753700,0.349356,0.418053,0.462833,0.463710,0.073457
"(15000.0, inf]",11044,0.021503,23820.480769,14885.185347,9988.640724,0.400274,0.468238,0.583575,0.384372,0.032054


In [ ]:
#df_impl.to_parquet(ruta_base+"/Entregables_Validacion/Entregables_Iniciales/Bases/MALBIE2_RCC_202507.parquet",compression='GZIP',engine="pyarrow")

In [32]:
df = pd.read_parquet('C:/Users/s5226347/OneDrive - The Bank of Nova Scotia/Pedidos/023. Estimador_Ingresos_Cliente/Entregables_Validacion/Entregables_Iniciales/Bases/BASE_MALBIE2_OOT3.parquet',engine='pyarrow')

In [33]:
df.shape

(598432, 50)

In [34]:
#CONEXION AL SERVIDOR 13
server = 'CGSSCSF13\\SQL3,277'
bd = 'MODELOSH'
conexion = pyodbc.connect(driver='SQL server', host = server, database = bd)
print('Conexión exitosa')
query = f'''(
    SELECT ID_REGISTRO,INGRESO_MALBIE_V2,MAX_MAX_LINCRE_TOP4_12M
    FROM MODELOSH.DBO.MALBIE2_OOT_202508
) '''
df_out = pd.read_sql_query(query, conexion)
conexion.close()

Conexión exitosa


In [35]:
df.columns

Index(['ID_REGISTRO', 'RCC', 'COD_SBS_CLI', 'TIP_DOC_IDE', 'DOC_IDENTIDAD',
       'AFLUENTE_PROPUESTO_2026', 'UBIGEO_INMUEBLE', 'CARRERA',
       'GHIGHESTBAL_NORM48', 'MED_PAY_24M', 'MAX_CONSUMO_TOP4_60M',
       'MED_PASIVOS_IC_MIN6_12M', 'MONTO_TRANSACCIONES_12M',
       'MED_RED_CONSUMO_NREV_24M', 'MAX_MAX_LINCRE_TOP4_12M',
       'MAX_HIPOTECARIO_2010', 'AVG_RED_CONSUMO_TOTAL_24M', 'GAVGCCLIMIT1TO48',
       'ANTIGUEDAD_TCPP_144M', 'GBAL1PCTAVGBAL1TO18',
       'MED_RCC_S_TOT_U12M_6CONS', 'MAX_MNTO_ENV_PLIN_12M',
       'MAX_GRUPO_VEHICULO', 'MAX_GARHIP_120M', 'MAX_COMPRAS_TC_24M',
       'INGRESO_MALBIE23', 'INGRESO_CAP_V2', 'INGRESO_CAP_V2_LOG',
       'INGRESO_MALBIE_V2', 'INGRESO_MALBIE_V2_LOG',
       'shap_val_UBIGEO_INMUEBLE', 'shap_val_CARRERA',
       'shap_val_GHIGHESTBAL_NORM48', 'shap_val_MED_PAY_24M',
       'shap_val_MAX_CONSUMO_TOP4_60M', 'shap_val_MED_PASIVOS_IC_MIN6_12M',
       'shap_val_MONTO_TRANSACCIONES_12M', 'shap_val_MED_RED_CONSUMO_NREV_24M',
       'shap

In [37]:
#df = df.rename(columns={'MAX_MAX_LINCRE_TOP4_24M':'MAX_MAX_LINCRE_TOP4_12M'})
cols = df.columns
df = df.drop(columns=['INGRESO_MALBIE_V2','MAX_MAX_LINCRE_TOP4_12M'])
df = df.merge(df_out,how='left',on='ID_REGISTRO')
df = df[cols]
df['INGRESO_MALBIE_V2_LOG'] = np.log(df['INGRESO_MALBIE_V2'])

In [38]:
df.shape

(598432, 50)

In [39]:
df.head(5)

,ID_REGISTRO,RCC,COD_SBS_CLI,TIP_DOC_IDE,DOC_IDENTIDAD,AFLUENTE_PROPUESTO_2026,UBIGEO_INMUEBLE,CARRERA,GHIGHESTBAL_NORM48,MED_PAY_24M,...,shap_val_AVG_RED_CONSUMO_TOTAL_24M,shap_val_GAVGCCLIMIT1TO48,shap_val_ANTIGUEDAD_TCPP_144M,shap_val_GBAL1PCTAVGBAL1TO18,shap_val_MED_RCC_S_TOT_U12M_6CONS,shap_val_MAX_MNTO_ENV_PLIN_12M,shap_val_MAX_GRUPO_VEHICULO,shap_val_MAX_GARHIP_120M,shap_val_MAX_COMPRAS_TC_24M,expected_value
0,404213,202508,81056102,1,42468719,4. Personal,0150130,Missing,51650.000000,1173.61,...,0.045281,-0.034657,0.005735,0.001866,0.034327,0.003528,-0.016856,-0.008223,-0.017816,7.788703
1,72083,202508,181996315,1,07708495,4. Personal,0150120,Missing,NaN,NaN,...,-0.040552,-0.008608,-0.032724,-0.008404,-0.005126,-0.016795,-0.028471,-0.005870,-0.010276,7.788703
2,458020,202508,88014987,1,43691825,4. Personal,0150101,FINANZAS,6306.940000,700.00,...,0.039627,-0.004234,0.004489,-0.001049,-0.000534,0.056723,-0.016061,-0.008100,0.003477,7.788703
3,73444,202508,59857096,1,07759205,5. Clásico,0150120,Missing,292.885626,7.40,...,-0.047777,-0.024630,0.045298,-0.003107,-0.005268,-0.018643,-0.020892,-0.006721,-0.005881,7.788703
4,694012,202508,201748933,1,71810678,5. Clásico,0060202,Missing,679.533062,NaN,...,-0.022961,-0.031798,-0.035340,-0.008900,-0.005358,-0.021700,-0.014616,-0.004890,-0.005076,7.788703


In [40]:
df.to_parquet('C:/Users/s5226347/OneDrive - The Bank of Nova Scotia/Pedidos/023. Estimador_Ingresos_Cliente/Entregables_Validacion/Entregables_Iniciales/Bases/BASE_MALBIE2_OOT3.parquet',engine='pyarrow',compression='GZIP')